## Verification du pipeline PostgreSQL

Controles de coherence entre les couches staging -> clean -> features -> predictions.
Ce notebook ne fait que LIRE la base (aucune ecriture) : il sert a explorer et
valider visuellement ce que produisent les scripts `database/*.py`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "database"))

import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import text
from db_connection import get_engine

pd.set_option("display.max_columns", 50)
engine = get_engine()

## 1. Vue d'ensemble : nombre de lignes par table (bronze -> silver -> gold)

Verifie que chaque etape du pipeline a bien ecrit ses tables et que les volumes
sont coherents d'une couche a l'autre (aucune perte massive inattendue).

In [ ]:
TABLES = {
    "Bronze (staging)": ["staging_taches_2024", "staging_taches_2025", "staging_agents", "staging_absences"],
    "Silver (clean)": ["taches_clean", "agents_clean", "absences_clean"],
    "Gold (features)": ["features_detail", "features_journalier", "features_journalier_service"],
    "Predictions (Phase 3)": ["predictions_volumes_j30", "predictions_charge_etp",
                              "predictions_risque_retard", "anomalies_detectees"],
}

with engine.connect() as conn:
    existantes = set(pd.read_sql(
        "SELECT table_name FROM information_schema.tables WHERE table_schema='public'", conn
    )["table_name"])

for couche, tables in TABLES.items():
    print(f"\n{couche}")
    for t in tables:
        if t in existantes:
            n = pd.read_sql(text(f"SELECT count(*) c FROM {t}"), engine)["c"].iloc[0]
            print(f"  {t:<32} : {n:>10,} lignes")
        else:
            print(f"  {t:<32} : (pas encore creee)")

## 2. Qualite des donnees clean : valeurs manquantes et bornes

`data_cleaning.py` doit avoir elimine les valeurs hors bornes (volumes > 500,
temps negatifs, dates incoherentes). On verifie ici qu'il n'en reste aucune.

In [ ]:
controle = pd.read_sql("""
    SELECT
      count(*)                                            AS total,
      count(*) FILTER (WHERE volume_dossiers > 500)        AS volumes_hors_borne,
      count(*) FILTER (WHERE temps_passe_declare_min < 0)  AS temps_negatifs,
      count(*) FILTER (WHERE date_cloture < date_creation) AS dates_incoherentes,
      count(*) FILTER (WHERE service_agent IS NULL)        AS taches_sans_agent,
      round(100.0 * count(*) FILTER (WHERE temps_passe_declare_min IS NULL) / count(*), 1) AS pct_temps_manquant,
      round(100.0 * count(*) FILTER (WHERE date_cloture IS NULL) / count(*), 1)            AS pct_cloture_manquante
    FROM taches_clean
""", engine)
controle

## 3. Coherence ETP (verification du bug historique corrige)

L'ancien pipeline Colab avait un bug d'inflation du calcul ETP du a une
deduplication manquante par tache. On verifie ici que la somme des ETP par
service correspond exactement a l'ETP global, jour par jour.

In [ ]:
etp_global = pd.read_sql("SELECT date, etp_disponible FROM features_journalier ORDER BY date", engine)
etp_service = pd.read_sql("""
    SELECT date, sum(etp_disponible) AS etp_somme_service
    FROM features_journalier_service GROUP BY date ORDER BY date
""", engine)

comparaison_etp = etp_global.merge(etp_service, on="date")
comparaison_etp["ecart"] = (comparaison_etp["etp_disponible"] - comparaison_etp["etp_somme_service"]).abs()

print(f"Ecart maximum observe : {comparaison_etp['ecart'].max():.6f}")
print(f"Jours avec ecart > 0.01 : {(comparaison_etp['ecart'] > 0.01).sum()} / {len(comparaison_etp)}")
comparaison_etp.head()

## 4. Visualisation : volume entrant journalier

In [ ]:
df_journalier = pd.read_sql("SELECT * FROM features_journalier ORDER BY date", engine)

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df_journalier["date"], df_journalier["volume_entrant_jour"], color="#2c7bb6", linewidth=1)
ax.plot(df_journalier["date"], df_journalier["volume_moy_7j"], color="#d7191c", linewidth=1.5, label="moyenne mobile 7j")
ax.set_title("Volume entrant journalier")
ax.set_xlabel("Date")
ax.set_ylabel("Dossiers / jour")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Distribution de la charge par ETP et taux d'alerte de surcharge par service

In [ ]:
df_service = pd.read_sql("SELECT * FROM features_journalier_service", engine)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].hist(df_journalier["charge_par_etp"].dropna(), bins=40, color="#2c7bb6")
axes[0].set_title("Distribution charge_par_etp (global)")
axes[0].set_xlabel("Dossiers / ETP")

taux_alerte_service = df_service.groupby("Service")["alerte_surcharge"].mean().sort_values(ascending=False) * 100
taux_alerte_service.plot(kind="barh", ax=axes[1], color="#d7191c")
axes[1].set_title("Taux de jours en alerte surcharge par service (%)")
axes[1].set_xlabel("% de jours en alerte")

plt.tight_layout()
plt.show()

## 6. Bloc 3A - Previsions de volumes (J+1 a J+30)

Modele retenu : SARIMA (Prophet indisponible sur cet environnement Windows,
cf. `models/model_volumes.pkl` pour le motif complet).

In [ ]:
df_prev_volumes = pd.read_sql("SELECT * FROM predictions_volumes_j30 ORDER BY date", engine)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df_prev_volumes["date"], df_prev_volumes["volume_prevu"], color="#2c7bb6", marker="o", markersize=3)
ax.fill_between(df_prev_volumes["date"], df_prev_volumes["volume_prevu_min"],
                 df_prev_volumes["volume_prevu_max"], color="#2c7bb6", alpha=0.2)
ax.set_title("Previsions de volume entrant, J+1 a J+30 (SARIMA)")
ax.set_xlabel("Date")
ax.set_ylabel("Volume prevu")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

df_prev_volumes.head(10)

## 7. Bloc 3B - Charge par ETP : reel vs predit (test set)

In [ ]:
df_prev_charge = pd.read_sql("SELECT * FROM predictions_charge_etp ORDER BY date", engine)
df_prev_charge_test = df_prev_charge[df_prev_charge["split"] == "test"]

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df_prev_charge_test["date"], df_prev_charge_test["charge_reelle"], label="Reel", color="#2c7bb6")
ax.plot(df_prev_charge_test["date"], df_prev_charge_test["charge_predite"], label="Predit", color="#d7191c", linestyle="--")
ax.set_title(f"Charge par ETP - Reel vs Predit (test, modele={df_prev_charge['modele'].iloc[0]})")
ax.set_xlabel("Date")
ax.set_ylabel("Dossiers / ETP")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Bloc 3C - Risque de retard : distribution des scores

In [ ]:
stats_retard = pd.read_sql("""
    SELECT
      count(*)                                              AS total,
      round(100.0 * avg(est_en_retard_reel::int), 1)         AS pct_retard_reel,
      round(100.0 * avg(est_en_retard_predit::int), 1)       AS pct_retard_predit,
      round(avg(score_risque_retard)::numeric, 3)            AS score_moyen
    FROM predictions_risque_retard
""", engine)
print(stats_retard)

echantillon_scores = pd.read_sql(
    "SELECT score_risque_retard FROM predictions_risque_retard ORDER BY random() LIMIT 50000", engine
)
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(echantillon_scores["score_risque_retard"], bins=40, color="#2c7bb6")
ax.set_title("Distribution du score de risque de retard (echantillon)")
ax.set_xlabel("Score (probabilite)")
plt.tight_layout()
plt.show()

## 9. Bloc 3D - Anomalies detectees

In [ ]:
taux_anomalies = pd.read_sql("""
    SELECT "Service", round(100.0 * avg(est_anomalie::int), 2) AS pct_anomalies, count(*) AS total
    FROM anomalies_detectees GROUP BY "Service" ORDER BY pct_anomalies DESC
""", engine)
print(taux_anomalies)

top_anomalies = pd.read_sql("""
    SELECT "ID_Tache", "Matricule_Agent", "Service", "Type_Processus",
           "Temps_Passe_Declare_Min", "Volume_Dossiers", "productivite_dossiers_par_heure", score_anomalie
    FROM anomalies_detectees
    WHERE est_anomalie = 1
    ORDER BY score_anomalie ASC
    LIMIT 10
""", engine)
top_anomalies